# 📈 คำอธิบายและตัวอย่างการปฏิบัติการการถดถอยเชิงเส้น (Linear Regression)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **การถดถอยเชิงเส้น (Linear Regression)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลอง (Synthetic Dataset) ที่แสดงความสัมพันธ์เชิงเส้นแบบมีสัญญาณรบกวน (Noise)
2. สร้างแบบจำลอง **Linear Regression** โดยใช้ไลบรารีมาตรฐาน `scikit-learn`
3. สร้างแบบจำลอง **Linear Regression จากศูนย์ (from scratch)** ด้วยวิธี **Gradient Descent** เพื่อทำความเข้าใจคณิตศาสตร์เบื้องหลัง
4. จำลองภาพกระบวนการฝึกสอน (Training Process) และเส้นถดถอยที่ดีที่สุด (Best-fit line)
5. ประเมินประสิทธิภาพของแบบจำลองโดยใช้ตัววัดมาตรฐาน: Mean Squared Error (MSE) และ R-squared ($R^2$)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลจำลอง (Data Generation)

เราจะสร้างข้อมูลจำลองอ้างอิงตามความสัมพันธ์เชิงเส้น:
$$y = w \cdot x + b + \epsilon$$

โดยกำหนดให้:
*   ค่าน้ำหนักจริง (True Weight: $w$) = 2.5
*   ค่าอคติจริง (True Bias: $b$) = 1.0
*   $\epsilon$ คือสัญญาณรบกวนแบบเกาส์เซียน (Gaussian Noise) ที่จำลองความคลาดเคลื่อนในการวัดค่าในโลกจริง

In [ ]:
# สุ่มสร้างจุดข้อมูล 100 จุดที่มีค่าระหว่าง 0 ถึง 5
X = np.random.rand(100, 1) * 5
true_w = 2.5
true_b = 1.0
noise = np.random.randn(100, 1) * 0.8

y = true_w * X + true_b + noise

# พล็อตกราฟจุดข้อมูลที่ถูกสร้างขึ้น
plt.figure(figsize=(8, 5))
plt.scatter(X, y, color='blue', alpha=0.6, label='Data Points (y = 2.5x + 1 + noise)')
plt.xlabel('X (Independent Variable)')
plt.ylabel('y (Dependent Variable)')
plt.title('Synthetic Linear Data Generation')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.show()

## 2. การทำ Linear Regression โดยใช้ Scikit-Learn

เรามาลองเทรนแบบจำลองการถดถอยโดยใช้ไลบรารีมาตรฐานในอุตสาหกรรมอย่าง `scikit-learn` กันครับ โดยคลาส `LinearRegression` จะค้นหาเส้นที่ลดค่าความคลาดเคลื่อนกำลังสองน้อยที่สุด (Ordinary Least Squares)

In [ ]:
# สร้างแบบจำลองและเทรนข้อมูล (Fit)
model = LinearRegression()
model.fit(X, y)

# ดึงค่าพารามิเตอร์ที่แบบจำลองเรียนรู้ได้
learned_w = model.coef_[0][0]
learned_b = model.intercept_[0]

print(f"True Weight: {true_w} | Learned Weight: {learned_w:.4f}")
print(f"True Bias: {true_b}   | Learned Bias: {learned_b:.4f}")

# ทำการทำนายผล
y_pred_sklearn = model.predict(X)

## 3. การสร้าง Linear Regression จากศูนย์ (Custom Linear Regression from Scratch ด้วย Gradient Descent)

เพื่อที่จะทำความเข้าใจว่าค่าน้ำหนักถูกปรับค่าให้ดีที่สุดได้อย่างไร เราจะเขียนโค้ดสำหรับแบบจำลองการถดถอยเชิงเส้นขึ้นมาเองจากศูนย์ ทบทวนฟังก์ชันค่าใช้จ่าย (Cost Function - Mean Squared Error):
$$J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (h_{w,b}(x^{(i)}) - y^{(i)})^2$$

และกฎการปรับน้ำหนักด้วยเกรเดียนต์ (Gradient Updates):
$$w \leftarrow w - \alpha \frac{1}{m} \sum_{i=1}^{m} (h_{w,b}(x^{(i)}) - y^{(i)}) \cdot x^{(i)}$$
$$b \leftarrow b - \alpha \frac{1}{m} \sum_{i=1}^{m} (h_{w,b}(x^{(i)}) - y^{(i)})$$

มาเริ่มเขียนโค้ดทำงานวนซ้ำ (Iterative Process) ด้วย Python กันครับ

In [ ]:
# ไฮเปอร์พารามิเตอร์
learning_rate = 0.05
epochs = 200
m = len(X)

# ตั้งค่าเริ่มต้นให้น้ำหนัก (w) และอคติ (b) เป็น 0
w_custom = 0.0
b_custom = 0.0

# เก็บประวัติค่าวัดต่างๆ ไว้ทำกราฟจำลองภาพ
cost_history = []
w_history = []
b_history = []

for epoch in range(epochs):
    # การทำนายผลตามสมมติฐาน (Hypothesis)
    y_pred_custom = w_custom * X + b_custom
    
    # คำนวณค่าความคลาดเคลื่อนกำลังสองเฉลี่ย (MSE)
    error = y_pred_custom - y
    cost = (1 / (2 * m)) * np.sum(error ** 2)
    cost_history.append(cost)
    w_history.append(w_custom)
    b_history.append(b_custom)
    
    # คำนวณเกรเดียนต์ (ค่าความชันสำหรับปรับทิศทาง)
    dw = (1 / m) * np.sum(error * X)
    db = (1 / m) * np.sum(error)
    
    # อัปเดตค่าน้ำหนักและอคติ
    w_custom -= learning_rate * dw
    b_custom -= learning_rate * db
    
    # บันทึกความคืบหน้าทุกๆ 40 รอบการฝึกสอน (Epochs)
    if epoch % 40 == 0:
        print(f"Epoch {epoch:03d} | Cost: {cost:.4f} | w: {w_custom:.4f} | b: {b_custom:.4f}")

print(f"\nFinal learned weights from scratch:")
print(f"Learned Weight (w): {w_custom:.4f}")
print(f"Learned Bias (b): {b_custom:.4f}")

## 4. การจำลองภาพการลู่เข้าหาคำตอบของ Gradient Descent (Convergence Visualization)

เรามาดูกันว่าเส้นถดถอยมีการปรับตัวอย่างไรในแต่ละรอบการฝึกสอน (Epochs) ในตอนเริ่มต้น เส้นตรงจะเริ่มจากด้านล่างสุด ($w=0, b=0$) และจะค่อย ๆ ขยับปรับตัวเข้าหาแนวโน้มของจุดข้อมูลจริงอย่างสวยงามครับ

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='blue', alpha=0.5, label='Data Points')

# พล็อตเส้นถดถอยที่แต่ละขั้นการฝึกสอนที่กำหนด
epochs_to_plot = [0, 5, 15, 30, 80, 199]
colors = ['red', 'orange', 'gold', 'lightgreen', 'cyan', 'green']

for epoch, color in zip(epochs_to_plot, colors):
    w_t = w_history[epoch]
    b_t = b_history[epoch]
    y_line = w_t * X + b_t
    plt.plot(X, y_line, color=color, linestyle='--', linewidth=1.5,
             label=f'Epoch {epoch} (w={w_t:.2f}, b={b_t:.2f})')

plt.xlabel('X')
plt.ylabel('y')
plt.title('Gradient Descent: Evolution of Regression Line')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

นอกจากนี้ เรามาพล็อตกราฟค่าใช้จ่าย (Cost Curve/Loss Curve) เพื่อตรวจสอบว่ากระบวนการปรับค่าเหมาะสมเป็นไปอย่างราบรื่นและลู่เข้าหาคำตอบจริงหรือไม่

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(cost_history, color='purple', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Cost (MSE / 2)')
plt.title('Cost Reduction Over Training Epochs')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 5. ตัววัดประสิทธิภาพแบบจำลอง (Evaluation Metrics)

เพื่อวัดเป็นตัวเลขเชิงปริมาณว่าแบบจำลองการถดถอยของเราฟิตเข้ากับข้อมูลได้ดีเพียงใด เราจะใช้ตัววัดผล 2 ตัว:
1.  **Mean Squared Error (MSE)**: ค่าเฉลี่ยของกำลังสองของผลต่างระหว่างค่าทำนายและค่าจริง ยิ่งน้อยยิ่งแสดงว่าโมเดลแม่นยำ
2.  **Coefficient of Determination ($R^2$ Score)**: อัตราส่วนความแปรปรวนของตัวแปรตามที่สามารถอธิบายได้ด้วยตัวแปรอิสระ มีค่าระหว่าง 0 ถึง 1 (ค่าเป็น 1 หมายถึงแบบจำลองสามารถทำนายได้สมบูรณ์แบบไร้ความคลาดเคลื่อน)

In [ ]:
# การทำนายและวัดผลด้วย Scikit-Learn
y_pred_sklearn = model.predict(X)
mse_sklearn = mean_squared_error(y, y_pred_sklearn)
r2_sklearn = r2_score(y, y_pred_sklearn)

# การทำนายและวัดผลด้วยน้ำหนักที่เราเขียนขึ้นเองจากศูนย์
y_pred_scratch = w_custom * X + b_custom
mse_scratch = mean_squared_error(y, y_pred_scratch)
r2_scratch = r2_score(y, y_pred_scratch)

print("--- Scikit-Learn Metrics ---")
print(f"MSE: {mse_sklearn:.4f}")
print(f"R2 Score: {r2_sklearn:.4f}")

print("\n--- Custom Scratch Metrics ---")
print(f"MSE: {mse_scratch:.4f}")
print(f"R2 Score: {r2_scratch:.4f}")

## 💡 ความเชื่อมโยงสู่ Deep Learning และ Computer Vision
*   ในงาน **ตรวจจับวัตถุ (เช่น YOLO)** การทำนายพิกัดตำแหน่งกล่องข้อความ (Bounding Box Coordinates: $x, y, w, h$) ถือเป็นปัญหาประเภท **การถดถอย (Regression)**
*   แทนที่จะใช้เส้นตรงง่ายๆ เครือข่ายประสาทระดับลึก (Deep Networks) จะเรียนรู้ฟังก์ชันการถดถอยที่มีความไม่เป็นเชิงเส้นสูงมาก (Highly Non-linear Regression)
*   แทนที่จะปรับจูนแค่ 2 ตัวแปร ($w$ และ $b$) เครือข่ายประสาทระดับลึกจะปรับแต่งพารามิเตอร์นับล้านตัว โดยใช้กระบวนการส่งย้อนกลับ (Backpropagation) ในการหาค่าเกรเดียนต์ ซึ่งมีแนวคิดเช่นเดียวกันกับการหาอนุพันธ์ที่เราลงมือคำนวณข้างต้นนี้เองครับ